# Notebook de Machine Learning — MVP Interações Medicamentosas

Este notebook implementa um pipeline completo de Machine Learning para **classificar a gravidade de interações medicamentosas** (DDI) em um cenário hospitalar (pacientes internados).

A estrutura segue o estilo do projeto **pharm-mvp**: seções numeradas, carregamento de dataset público, EDA, pré-processamento, baseline + 2 modelos, avaliação, comparação e validação qualitativa com referências brasileiras.

## 1) Descrição do Problema

- **Objetivo:** classificar a gravidade de interações medicamentosas (Leve/Moderada/Grave) para apoiar segurança do paciente internado.
- **Tipo de problema:** classificação supervisionada multiclasse.
- **Variável alvo:** gravidade da interação (inferida a partir da descrição textual do dataset).
- **Observação importante (evitar vazamento):** como o rótulo é derivado da descrição, **não usamos a descrição como feature** no treinamento — usamos apenas o par de medicamentos.

## 2) Datasets Públicos

### Dataset internacional (base de treino)
- **Fonte:** Kaggle — Drug-Drug Interactions (derivado do DrugBank / TDCommons).
- **Link:** https://www.kaggle.com/datasets/mghobashy/drug-drug-interactions
- **Arquivo:** `db_drug_interactions.csv` (3 colunas: `Drug 1`, `Drug 2`, `Interaction Description`).
- **Licença:** Apache 2.0.

### Dados brasileiros (validação cruzada / comparação)
- **Situação:** não há um dataset tabular brasileiro aberto e padronizado de DDI com gravidade para ML.
- **Estratégia:** validação **qualitativa** com uma lista (curada) de interações clinicamente relevantes e frequentemente citadas em protocolos/rotinas no Brasil, + opção de carregar uma lista brasileira em CSV caso você tenha uma fonte institucional/local.

In [ ]:
# 3) Importação de bibliotecas
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
plt.style.use("ggplot")

## 4) Carregamento do Dataset Público

1. Baixe o dataset no Kaggle (link na Seção 2).
2. Coloque o arquivo `db_drug_interactions.csv` em `pharmIA/data/` (**recomendado**) ou na mesma pasta do notebook.


In [ ]:
# Procura primeiro em data/ (padrão do projeto), depois na pasta atual
candidates = [Path('data') / 'db_drug_interactions.csv', Path('db_drug_interactions.csv')]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        'db_drug_interactions.csv não encontrado. Coloque em pharmIA/data/ (recomendado) ou na pasta do notebook.'
    )

df_raw = pd.read_csv(data_path)
print('Arquivo:', data_path.resolve())
print('Shape:', df_raw.shape)
df_raw.head()

## 5) Análise Exploratória dos Dados (EDA)

Exploramos estrutura, valores ausentes, principais medicamentos e características básicas do texto de descrição.

In [ ]:
df_raw.info()

display(df_raw.isna().sum())

display(df_raw['Drug 1'].value_counts().head(10))
display(df_raw['Drug 2'].value_counts().head(10))

# Tamanho das descrições (proxy simples de complexidade do texto)
desc_len = df_raw['Interaction Description'].astype(str).str.len()
print('Comprimento da descrição (min/med/max):', desc_len.min(), desc_len.median(), desc_len.max())

df_raw['Interaction Description'].sample(5, random_state=RANDOM_STATE)

## 6) Pré-processamento dos Dados

### 6.1) Rotulagem de gravidade (regra baseada em palavras-chave)
Como o dataset não traz a gravidade explicitamente, criamos o rótulo `severity` a partir da descrição (`Interaction Description`).

### 6.2) Features
Para evitar vazamento, treinaremos os modelos usando **somente** o par de medicamentos (`Drug 1`, `Drug 2`).

### 6.3) Codificação
Usaremos **One-Hot Encoding** para tratar as variáveis categóricas corretamente (evita o problema de `LabelEncoder` introduzir ordem artificial).

In [ ]:
def classify_severity(text: str) -> str:
    if not isinstance(text, str):
        text = ''
    t = text.lower()

    # Heurística simples: ajuste conforme necessidade
    if re.search(r'contraindicat|life[- ]?threat|fatal|severe', t):
        return 'Grave'
    if re.search(r'moderate|monitor|caution|dose adjustment|avoid concomitant', t):
        return 'Moderada'
    if re.search(r'mild|minor|no significant|not clinically significant', t):
        return 'Leve'
    return 'Moderada'  # padrão conservador

df = df_raw.copy()
df['severity'] = df['Interaction Description'].apply(classify_severity)

# Limpeza mínima
df = df.dropna(subset=['Drug 1', 'Drug 2', 'Interaction Description'])
df['drug_1'] = df['Drug 1'].astype(str).str.strip().str.lower()
df['drug_2'] = df['Drug 2'].astype(str).str.strip().str.lower()

print('Shape após limpeza:', df.shape)
display(df['severity'].value_counts())

# Split
X = df[['drug_1', 'drug_2']]
y = df['severity']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

categorical_features = ['drug_1', 'drug_2']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[('cat', categorical_transformer, categorical_features)],
    remainder='drop'
)

In [ ]:
# Verificar distribuição das classes após split
print('Distribuição em y_train:')
print(y_train.value_counts())
print('Distribuição em y_test:')
print(y_test.value_counts())

## 7) Treinamento do Modelo Baseline

Baseline: modelo que sempre prevê a classe mais frequente. Serve como referência mínima de desempenho.

In [ ]:
baseline_model = DummyClassifier(strategy='most_frequent')
baseline_pipe = Pipeline(steps=[('preprocess', preprocess), ('model', baseline_model)])

baseline_pipe.fit(X_train, y_train)
pred_base = baseline_pipe.predict(X_test)

metrics = []
metrics.append({
    'Modelo': 'Baseline (Most frequent)',
    'Accuracy': accuracy_score(y_test, pred_base),
    'F1_weighted': f1_score(y_test, pred_base, average='weighted'),
    'F1_macro': f1_score(y_test, pred_base, average='macro'),
})

pd.DataFrame(metrics)

## 8) Treinamento de Dois Modelos Adicionais

Treinaremos dois modelos clássicos adicionais e compararemos com o baseline:
- Regressão Logística (multiclasse)
- Naive Bayes Multinomial (adequado a features one-hot)

In [ ]:
# Modelo 1: Regressão Logística
logreg = LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=RANDOM_STATE)
logreg_pipe = Pipeline(steps=[('preprocess', preprocess), ('model', logreg)])
logreg_pipe.fit(X_train, y_train)
pred_log = logreg_pipe.predict(X_test)

metrics.append({
    'Modelo': 'LogisticRegression',
    'Accuracy': accuracy_score(y_test, pred_log),
    'F1_weighted': f1_score(y_test, pred_log, average='weighted'),
    'F1_macro': f1_score(y_test, pred_log, average='macro'),
})

# Modelo 2: Multinomial Naive Bayes
nb = MultinomialNB()
nb_pipe = Pipeline(steps=[('preprocess', preprocess), ('model', nb)])
nb_pipe.fit(X_train, y_train)
pred_nb = nb_pipe.predict(X_test)

metrics.append({
    'Modelo': 'MultinomialNB',
    'Accuracy': accuracy_score(y_test, pred_nb),
    'F1_weighted': f1_score(y_test, pred_nb, average='weighted'),
    'F1_macro': f1_score(y_test, pred_nb, average='macro'),
})

results = pd.DataFrame(metrics).sort_values('F1_weighted', ascending=False)
results

## 9) Avaliação dos Modelos (múltiplas métricas)

Métricas usadas por algoritmo (mínimo 2):
- **Accuracy**
- **F1-score (weighted)**
- (extra) **F1-score (macro)**

Também exibimos matriz de confusão para análise por classe.

In [ ]:
labels = sorted(y.unique().tolist())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, pred, name in zip(
    axes,
    [pred_base, pred_log, pred_nb],
    ['Baseline', 'LogisticRegression', 'MultinomialNB']
):
    cm = confusion_matrix(y_test, pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Matriz de Confusão — {name}')

plt.tight_layout()
plt.show()

## 10) Comparação dos Resultados

Comparamos os modelos lado a lado em tabela e gráfico.

In [ ]:
display(results)

ax = results.set_index('Modelo')[['Accuracy', 'F1_weighted', 'F1_macro']].plot(
    kind='bar', figsize=(10, 5), ylim=(0, 1), title='Comparação dos Modelos'
)
ax.set_ylabel('Pontuação')
plt.xticks(rotation=0)
plt.show()

## 11) Validação Cruzada com Dados Brasileiros (qualitativa)

Como não há um dataset brasileiro tabular aberto e padronizado de DDI com gravidade, fazemos uma validação cruzada **qualitativa**:

1) Construímos uma lista **exemplificativa** de interações de alta relevância clínica frequentemente discutidas em rotinas hospitalares brasileiras (nomes INN/DCB).
2) Verificamos se essas interações aparecem no dataset e se foram rotuladas como **Grave** pela nossa heurística.

Opcional: se você tiver uma lista brasileira em CSV (por exemplo, extraída de protocolo institucional), você pode carregá-la e repetir o mesmo procedimento.

In [ ]:
# Lista exemplificativa (INN/DCB). Ajuste conforme sua fonte local/ institucional.
br_pairs = [
    ('warfarin', 'amiodarone'),
    ('digoxin', 'verapamil'),
    ('clopidogrel', 'omeprazole'),
    ('simvastatin', 'clarithromycin'),
    ('carbamazepine', 'erythromycin'),
    ('lithium', 'hydrochlorothiazide'),
]

def norm_drug(s: str) -> str:
    return str(s).strip().lower()

br_pairs = [(norm_drug(a), norm_drug(b)) for a, b in br_pairs]

# Conjunto de interações do dataset (considerando ambas as ordens)
pairs_ds = set(zip(df['drug_1'], df['drug_2']))
pairs_ds |= set((b, a) for (a, b) in pairs_ds)

present_in_dataset = [p for p in br_pairs if p in pairs_ds]
print('Interações da lista brasileira que aparecem no dataset:', len(present_in_dataset))
display(present_in_dataset)

# Entre as que aparecem no dataset, verificar as rotuladas como Grave
df_pairs = df[['drug_1', 'drug_2', 'severity']].copy()
df_pairs['pair'] = list(zip(df_pairs['drug_1'], df_pairs['drug_2']))

grave_set = set(df_pairs.loc[df_pairs['severity'] == 'Grave', 'pair'])
grave_set |= set((b, a) for (a, b) in grave_set)

matches_grave = [p for p in br_pairs if p in grave_set]
print('Interações da lista brasileira rotuladas como Grave no dataset:', len(matches_grave))
display(matches_grave)

# (Opcional) carregar lista brasileira em CSV
csv_br = Path('interacoes_brasil.csv')
if csv_br.exists():
    df_br = pd.read_csv(csv_br)
    # Esperado: colunas drug_a, drug_b
    if {'drug_a', 'drug_b'}.issubset(df_br.columns):
        pairs_br_csv = [(norm_drug(a), norm_drug(b)) for a, b in zip(df_br['drug_a'], df_br['drug_b'])]
        pairs_br_csv = [p for p in pairs_br_csv if p[0] and p[1]]
        present_csv = [p for p in pairs_br_csv if p in pairs_ds]
        print('CSV BR: pares presentes no dataset:', len(present_csv))
    else:
        print('CSV encontrado, mas faltam colunas esperadas: drug_a, drug_b')
else:
    print('Nenhum interacoes_brasil.csv encontrado (opcional).')

## 12) Justificativa da Escolha do Modelo Final

Escolhemos o modelo final considerando principalmente **F1-score (weighted)**, pois há desbalanceamento de classes e queremos bom desempenho global, sem ignorar classes menos frequentes.

- Se a **Regressão Logística** vencer: vantagem de interpretabilidade e probabilidades (`predict_proba`), útil para triagem clínica.
- Se o **Naive Bayes** vencer: simplicidade, rapidez e boa performance em dados categóricos one-hot.

**Limitações:** rótulo de gravidade foi inferido por heurística; recomenda-se validar com farmacêutico clínico e, idealmente, uma lista brasileira institucional (quando disponível).

## 13) Referências

- Kaggle — Drug-Drug Interactions (mghobashy): https://www.kaggle.com/datasets/mghobashy/drug-drug-interactions
- TDCommons — DDI task: https://tdcommons.ai/multi_pred_tasks/ddi/

(Para validação brasileira: usar protocolos institucionais/hospitalares, publicações e materiais oficiais quando disponíveis publicamente.)